# 06 — Imágenes como secuencias: RNN, LSTM, GRU y atención

Representaremos cada imagen RGB64 como una secuencia de columnas para estudiar recurrencia. Cada paso recibe una columna y el estado resume los pasos anteriores. Compararemos RNN simple, LSTM y GRU, y añadiremos atención sobre los estados recurrentes.

La atención forma parte de la cobertura mínima del profesor. Las variantes bidireccionales, apiladas o CNN + RNN son ampliaciones opcionales, sujetas al coste disponible.

> **Plan pendiente de implementación.** Este cuaderno contiene explicaciones y propuestas. No contiene código ejecutado ni resultados de entrenamiento.

## Requisito confirmado por el correo del profesorado

RNN + atención forma parte del núcleo, junto con LSTM y GRU. Comparar un mismo encoder recurrente con pooling de estados frente a una agregación aprendida con atención; conservar datos, dimensiones y presupuesto y publicar el cambio de parámetros. Mostrar pesos de atención sobre posiciones y explicar sus límites como interpretación. Las variantes bidireccionales, capas apiladas y CNN + RNN siguen siendo ampliaciones opcionales. Este bloque está planificado, todavía sin resultados.

## Vía A · Recurrencia, LSTM/GRU y atención como parte del estudio mínimo

Interpretamos «LSTN» del apunte como **LSTM**. LSTM y GRU son unidades recurrentes: procesan una secuencia mediante estados y compuertas; no llevan necesariamente una capa de atención. En este proyecto, los pasos son filas o columnas de píxeles. En NLP serían vectores de tokens.

Ruta didáctica propuesta: RNN sencilla → LSTM y GRU → agregación con atención como parte del mínimo obligatorio. Comparar último estado o pooling con una atención sobre los estados de la secuencia. Self-attention puede añadirse para relacionar posiciones, pero no define una RNN ni sustituye automáticamente su recurrencia. Una CNN + LSTM/GRU sería otra híbrida opcional. En 08 se estudia el contraste con un Transformer que utiliza self-attention sin recurrencia en su arquitectura estándar.

El objetivo es entender memoria, orden, longitud de secuencia y coste. Un resultado inferior al de CNN no invalida el experimento. [Referencia para distinguir atención y recurrencia](https://arxiv.org/abs/1706.03762).

## Protocolo común del proyecto

Primero definimos las prendas que queremos clasificar. Después aplicamos los mismos controles de calidad a todas las fotos y conservamos **todas las que los superan**, aunque unas clases tengan más ejemplos. No se fija un número total ni se igualan las clases con cupos.

Las particiones se construyen sobre todos los grupos de producto elegibles. Los productos ya utilizados durante el desarrollo conservan su exposición: un producto de train no se convierte en un supuesto test nuevo. Los grupos nuevos completan aproximadamente 70/15/15 por clase. Se publican los recuentos reales.

**Antes de recortar train por coste, hay que medirlo.** El baseline usa todos los productos elegibles; 04 registra una época piloto y la estimación de tiempo/memoria. Si fuese necesario reducir datos en otro experimento, se justificaría con esa medición, solo en train y conservando validation/test. Una fotografía fuera de una muestra de cálculo no sería una fotografía de mala calidad.

La referencia es `config/experimental_protocol_v1.json`: macro-F1 primaria, accuracy y resultados por clase; pérdida inicial sin pesos, sin aumentos; normalización solo con train a la resolución de entrada. Tres semillas y coste computacional. Los pesos de clase se estudian como ablación: con frecuencias desiguales ya no suponemos que sean casi uno. El test predictivo se reserva hasta congelar las decisiones. Las métricas describen las 26 clases elegidas tras los controles, no todo el catálogo.

### H6 · Representación secuencial fijada y reproducible

Referencia: imagen RGB64, tensor C×H×W = 3×64×64. Se recorre de izquierda a derecha: 64 columnas, cada paso con 192 características (filas y canales). No se aplana toda la imagen como una secuencia de 12.288 escalares. LSTM/GRU procesan así una longitud razonable para el presupuesto local.

Ablación principal: filas, de arriba abajo, también 64 pasos de 192 características; cambiar solo la orientación y mantener tamaño oculto, capas, regla de parada y presupuesto. Una variante por parches de 8×8 produce 64 pasos con 192 valores por parche a esta resolución, pero cambia el vecindario representado: documentar orden raster y aplanado. Un encoder aprendido de parches añadiría parámetros y ya no sería el mismo contraste. Los píxeles se normalizan con train antes de construir la secuencia.

Registrar forma de entrada, orden, longitud, dimensión, parámetros, tiempo y memoria. Comparar con CNN/Dense RGB64 sobre los mismos IDs. La conclusión separará el efecto de recurrencia del de la representación; no se escogerá una secuencia deliberadamente desfavorable para RNN.

**Filtro y reutilización.** En 02 se aplica el mismo criterio visual fijo a train, validation y test; las fotos conservadas no cambian de partición. El test se reserva para evaluación predictiva, aunque haya pasado controles e inspección visual. Se guardarán resultados y predicciones para recalcular métricas sin repetir ajustes. Cualquier extracción o entrenamiento costoso nuevo requerirá una activación explícita; este cuaderno todavía no implementa ese flujo.

## Objetivos

- Mostrar la conversión RGB64 → 64 pasos de 192 valores y su orden de lectura.
- Explicar y comparar RNN simple, LSTM y GRU, incluyendo puertas y número de parámetros.
- Aplicar atención sobre los estados de una RNN y explicar cómo obtiene sus pesos.
- Comparar lectura por columnas y filas manteniendo los mismos productos y particiones.
- Interpretar las diferencias con Dense y CNN sin atribuirlas solo a la recurrencia cuando cambie también la representación.

## Pasos previstos

1. Visualizar cómo la imagen se convierte en secuencia.
2. Construir RNN simple, LSTM y GRU con presupuesto documentado.
3. Añadir atención y contrastarla con la agregación de estados sin atención.
4. Estudiar el orden de lectura como ablación controlada.
5. Comparar métricas, incertidumbre y coste; dejar las ampliaciones no ejecutadas identificadas como propuestas.

## Entradas, salidas y material de clase

| | |
|---|---|
| **Entradas** | Imágenes RGB64 de las particiones de 02; referencias de 04 y 05 |
| **Salidas** | Métricas, predicciones, curvas y configuraciones recurrentes en results/ |
| **Material de clase** | `PRDL_MLLB_RecurrentNetworks.pptx`, `MLLB_Embeddings_RNN_SentimentAnalysis.ipynb`, `TO_DO_MLLB_Embeddings_TwoLayers_RNN_SentimentAnalysis.ipynb`, guideline slide "Image: RNN (LSTM, GRU…)" |